# 03. Non-Negative Matrix Factorization (NMF)

**Goal:** Train an NMF model using TF-IDF and compare the generated topics to LDA. NMF often performs exceptionally well on short texts like headlines.

In [1]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

## 1. Load Data

In [2]:
df = pd.read_csv('../data/processed/processed_headlines.csv')
df = df.dropna(subset=['lemmatized'])

> **📌 Decision Note — Why Vectorization for NMF?**
>
> **Chosen approach:** TfidfVectorizer
>
> **Why this works:** NMF operates via linear algebra (matrix decomposition), not probability. It factorizes the TF-IDF matrix into two lower-dimensional matrices (Topics and Weights).
>
> **Alternatives we could have used:**
> | Option | Pros | Cons |
> |--------|------|------|
> | CountVectorizer | Simple | Doesn't downweight overly common words, causing NMF topics to be noisy. |
> | LDA | Standard statistical approach | LDA often struggles with extremely short texts (like 5-word headlines) because there isn't enough word co-occurrence within a single document. |
>
> **Why we chose this over alternatives:** NMF paired with TF-IDF is practically the industry standard for extracting topics from short texts (Headlines, Tweets).

## 2. TF-IDF Vectorization

In [3]:
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, max_features=5000)
tfidf = tfidf_vectorizer.fit_transform(df['lemmatized'])
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

## 3. Fit NMF Model

In [4]:
nmf = NMF(n_components=10, random_state=42, init='nndsvd')
nmf.fit(tfidf)

,"n_components n_components: int or {'auto'} or None, default='auto'Number of components. If `None`, all features are kept.If `n_components='auto'`, the number of components is automatically inferredfrom W or H shapes... versionchanged:: 1.4 Added `'auto'` value... versionchanged:: 1.6 Default value changed from `None` to `'auto'`.",10
,"init init: {'random', 'nndsvd', 'nndsvda', 'nndsvdar', 'custom'}, default=NoneMethod used to initialize the procedure.Valid options:- `None`: 'nndsvda' if n_components <= min(n_samples, n_features), otherwise random.- `'random'`: non-negative random matrices, scaled with: `sqrt(X.mean() / n_components)`- `'nndsvd'`: Nonnegative Double Singular Value Decomposition (NNDSVD) initialization (better for sparseness)- `'nndsvda'`: NNDSVD with zeros filled with the average of X (better when sparsity is not desired)- `'nndsvdar'` NNDSVD with zeros filled with small random values (generally faster, less accurate alternative to NNDSVDa for when sparsity is not desired)- `'custom'`: Use custom matrices `W` and `H` which must both be provided... versionchanged:: 1.1 When `init=None` and n_components is less than n_samples and n_features defaults to `nndsvda` instead of `nndsvd`.",'nndsvd'
,"random_state random_state: int, RandomState instance or None, default=NoneUsed for initialisation (when ``init`` == 'nndsvdar' or'random'), and in Coordinate Descent. Pass an int for reproducibleresults across multiple function calls.See :term:`Glossary <random_state>`.",42
,"solver solver: {'cd', 'mu'}, default='cd'Numerical solver to use:- 'cd' is a Coordinate Descent solver.- 'mu' is a Multiplicative Update solver... versionadded:: 0.17 Coordinate Descent solver... versionadded:: 0.19 Multiplicative Update solver.",'cd'
,"beta_loss beta_loss: float or {'frobenius', 'kullback-leibler', 'itakura-saito'}, default='frobenius'Beta divergence to be minimized, measuring the distance between Xand the dot product WH. Note that values different from 'frobenius'(or 2) and 'kullback-leibler' (or 1) lead to significantly slowerfits. Note that for beta_loss <= 0 (or 'itakura-saito'), the inputmatrix X cannot contain zeros. Used only in 'mu' solver... versionadded:: 0.19",'frobenius'
,"tol tol: float, default=1e-4Tolerance of the stopping condition.",0.0001
,"max_iter max_iter: int, default=200Maximum number of iterations before timing out.",200
,"alpha_W alpha_W: float, default=0.0Constant that multiplies the regularization terms of `W`. Set it to zero(default) to have no regularization on `W`... versionadded:: 1.0",0.0
,"alpha_H alpha_H: float or ""same"", default=""same""Constant that multiplies the regularization terms of `H`. Set it to zero tohave no regularization on `H`. If ""same"" (default), it takes the same value as`alpha_W`... versionadded:: 1.0",'same'
,"l1_ratio l1_ratio: float, default=0.0The regularization mixing parameter, with 0 <= l1_ratio <= 1.For l1_ratio = 0 the penalty is an elementwise L2 penalty(aka Frobenius Norm).For l1_ratio = 1 it is an elementwise L1 penalty.For 0 < l1_ratio < 1, the penalty is a combination of L1 and L2... versionadded:: 0.17 Regularization parameter *l1_ratio* used in the Coordinate Descent solver.",0.0
,"verbose verbose: int, default=0Whether to be verbose.",0


## 4. Extract and Display Topics

In [5]:
def display_topics(model, feature_names, no_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print(f'Topic {topic_idx}:')
        print(' '.join([feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]))

display_topics(nmf, tfidf_feature_names, 10)

Topic 0:
interview extended michael nrl james steve ben luke matt john
Topic 1:
man charged murder court face charge death jailed accused missing
Topic 2:
council plan call govt back water health urged change cut
Topic 3:
police probe investigate search officer missing death hunt arrest seek
Topic 4:
new year zealand law get home set open york centre
Topic 5:
abc rural news national market weather business nsw sport analysis
Topic 6:
say report union trump expert minister labor need could must
Topic 7:
crash car dy killed fatal driver woman road two plane
Topic 8:
australia win day world cup australian south open first final
Topic 9:
fire house home school sydney crew warning blaze damage destroys


In [6]:
joblib.dump(nmf, '../models/nmf_model.pkl')
joblib.dump(tfidf_vectorizer, '../models/tfidf_vectorizer.pkl')

['../models/tfidf_vectorizer.pkl']

## Key Takeaways
- [x] Compare these topics to the LDA topics. You will generally find NMF topics on short headlines to be more coherent and distinct!